In [ ]:
import pandas as pd

orders = pd.read_csv('../data/orders.csv')

orders.head()

# 전체적인 데이터 구조 확인
orders.info()

In [ ]:
# 필요한 라이브러리 불러오기
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib


print('라이브러리 불러오기 완료!')

# 데이터 불러오기
df = pd.read_csv('../data/orders.csv')

# 데이터가 몇 행 몇 열인지 확인
print('데이터 크기:', df.shape)

# 앞에 5줄만 확인
print(df.head())

# 결측값(빈 값) 확인 - 비어있는 데이터가 있는지 체크
print('결측값 개수:')
print(df.isnull().sum())

# =============================================
# 1. 요일별 주문량 분석
# =============================================

# 요일 숫자를 이름으로 바꾸는 딕셔너리
# 0 = 일요일, 1 = 월요일, ... 6 = 토요일
요일이름 = {0: '일요일', 1: '월요일', 2: '화요일', 3: '수요일',
           4: '목요일', 5: '금요일', 6: '토요일'}

# order_dow 컬럼에 요일 이름 컬럼 새로 추가
df['요일'] = df['order_dow'].map(요일이름)

# 요일별로 주문 횟수 세기
요일별주문수 = df['요일'].value_counts()

# 일~토 순서로 정렬
요일순서 = ['일요일', '월요일', '화요일', '수요일', '목요일', '금요일', '토요일']
요일별주문수 = 요일별주문수.reindex(요일순서)

print('요일별 주문 횟수:')
print(요일별주문수)
print()
print('주문이 가장 많은 요일:', 요일별주문수.idxmax())
print('주문이 가장 적은 요일:', 요일별주문수.idxmin())

# 요일별 막대그래프 그리기
# 주말은 빨간색, 평일은 파란색으로 구분
색깔리스트 = []
for 요일 in 요일순서:
    if 요일 == '일요일' or 요일 == '토요일':
        색깔리스트.append('tomato')     # 주말 -> 빨간색
    else:
        색깔리스트.append('steelblue')  # 평일 -> 파란색

plt.figure(figsize=(10, 5))
plt.bar(요일순서, 요일별주문수.values, color=색깔리스트)

plt.title('요일별 주문량', fontsize=15)
plt.xlabel('요일')
plt.ylabel('주문 횟수')

# 막대 위에 숫자 표시
for i in range(len(요일순서)):
    숫자 = 요일별주문수.values[i]
    plt.text(i, 숫자 + 500, str(숫자), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('graph1_요일별주문량.png', dpi=150)
plt.show()
print('그래프 저장 완료!')

# =============================================
# 2. 시간대별 주문량 분석
# =============================================

# 시간대별 주문 횟수 세기 (0시 ~ 23시)
시간대별주문수 = df['order_hour_of_day'].value_counts().sort_index()

print('시간대별 주문 횟수:')
print(시간대별주문수)
print()
print('주문이 가장 많은 시간:', 시간대별주문수.idxmax(), '시')
print('주문이 가장 적은 시간:', 시간대별주문수.idxmin(), '시')

# 시간대별 꺾은선 그래프 그리기
피크시간 = 시간대별주문수.idxmax()

plt.figure(figsize=(12, 5))
plt.plot(시간대별주문수.index, 시간대별주문수.values, marker='o', color='steelblue', linewidth=2)

# 피크 시간 빨간 점선으로 표시
plt.axvline(x=피크시간, color='red', linestyle='--', label='피크: ' + str(피크시간) + '시')

# 새벽 시간대 회색 음영 처리
plt.axvspan(0, 6, alpha=0.1, color='gray', label='새벽(0~6시)')

plt.title('시간대별 주문량', fontsize=15)
plt.xlabel('시간 (0~23시)')
plt.ylabel('주문 횟수')
plt.xticks(range(0, 24))
plt.legend()
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('graph2_시간대별주문량.png', dpi=150)
plt.show()
print('그래프 저장 완료!')

# =============================================
# 3. 재구매 주기 분석
# =============================================

# 첫 주문은 이전 주문이 없어서 빈 값 -> 제외하고 분석
재구매데이터 = df['days_since_prior_order'].dropna()

print('전체 주문 수:', len(df))
print('재구매 주문 수 (첫 주문 제외):', len(재구매데이터))
print()
print('평균 재구매 주기:', round(재구매데이터.mean(), 1), '일')
print('중앙값 재구매 주기:', 재구매데이터.median(), '일')
print('가장 많은 재구매 주기:', 재구매데이터.mode()[0], '일')

# 재구매 주기 히스토그램 그리기
평균값 = 재구매데이터.mean()

plt.figure(figsize=(12, 5))
plt.hist(재구매데이터, bins=30, color='steelblue', edgecolor='white')

# 평균값 빨간 점선으로 표시
plt.axvline(x=평균값, color='red', linestyle='--', linewidth=2,
            label='평균: ' + str(round(평균값, 1)) + '일')

plt.title('재구매 주기 분포', fontsize=15)
plt.xlabel('이전 주문으로부터 경과일 수')
plt.ylabel('주문 횟수')
plt.legend(fontsize=12)
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('graph3_재구매주기분포.png', dpi=150)
plt.show()
print('그래프 저장 완료!')

# =============================================
# 4. 유저별 평균 재구매 주기 통계
# =============================================

# user_id 기준으로 묶어서 평균 재구매 주기 계산
유저별재구매주기 = df.groupby('user_id')['days_since_prior_order'].mean()

print('유저별 평균 재구매 주기 통계:')
print(round(유저별재구매주기.describe(), 1))
print()
print('전체 유저 수:', len(유저별재구매주기), '명')
print('유저들이 평균적으로 재구매하는 주기:', round(유저별재구매주기.mean(), 1), '일')




In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlibw

matplotlib.rc('font', family='Malgun Gothic')
matplotlib.rc('axes', unicode_minus=False)

# 데이터 불러오기
df = pd.read_csv('../data/orders.csv')
print('데이터 크기:', df.shape)
print(df.head())

# =============================================
# 1. 요일별 주문량
# =============================================

요일이름 = {0: '일요일', 1: '월요일', 2: '화요일', 3: '수요일',
           4: '목요일', 5: '금요일', 6: '토요일'}
요일순서 = ['일요일', '월요일', '화요일', '수요일', '목요일', '금요일', '토요일']

df['요일'] = df['order_dow'].map(요일이름)
요일별주문수 = df['요일'].value_counts().reindex(요일순서)

# 주말 빨간색, 평일 파란색
색깔리스트 = []
for 요일 in 요일순서:
    if 요일 == '일요일' or 요일 == '토요일':
        색깔리스트.append('tomato')
    else:
        색깔리스트.append('steelblue')

plt.figure(figsize=(10, 5))
plt.bar(요일순서, 요일별주문수.values, color=색깔리스트)
plt.title('요일별 주문량', fontsize=15)
plt.xlabel('요일')
plt.ylabel('주문 횟수')
plt.tight_layout()
plt.savefig('graph1_요일별주문량.png', dpi=150)
plt.show()

# =============================================
# 2. 시간대별 주문량
# =============================================

시간대별주문수 = df['order_hour_of_day'].value_counts().sort_index()
피크시간 = 시간대별주문수.idxmax()

plt.figure(figsize=(12, 5))
plt.plot(시간대별주문수.index, 시간대별주문수.values, marker='o', color='steelblue', linewidth=2)
plt.axvline(x=피크시간, color='red', linestyle='--', label='피크: ' + str(피크시간) + '시')
plt.axvspan(0, 6, alpha=0.1, color='gray', label='새벽(0~6시)')
plt.title('시간대별 주문량', fontsize=15)
plt.xlabel('시간 (0~23시)')
plt.ylabel('주문 횟수')
plt.xticks(range(0, 24))
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('graph2_시간대별주문량.png', dpi=150)
plt.show()

# =============================================
# 3. 재구매 주기
# =============================================

재구매데이터 = df['days_since_prior_order'].dropna()
평균값 = 재구매데이터.mean()

print('평균 재구매 주기:', round(평균값, 1), '일')
print('중앙값:', 재구매데이터.median(), '일')
print('최빈값:', 재구매데이터.mode()[0], '일')

plt.figure(figsize=(12, 5))
plt.hist(재구매데이터, bins=30, color='steelblue', edgecolor='white')
plt.axvline(x=평균값, color='red', linestyle='--', linewidth=2,
            label='평균: ' + str(round(평균값, 1)) + '일')
plt.title('재구매 주기 분포', fontsize=15)
plt.xlabel('이전 주문으로부터 경과일 수')
plt.ylabel('주문 횟수')
plt.legend(fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('graph3_재구매주기분포.png', dpi=150)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 직접 경로로 설정
font_path = 'C:/Windows/Fonts/malgun.ttf'
font = fm.FontProperties(fname=font_path)
font_name = font.get_name()

plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False

# 불러오기
df = pd.read_csv('../data/orders.csv')
print('데이터 크기:', df.shape)
print(df.head())

# =============================================
# 1. 요일별 주문량
# =============================================

요일이름 = {0: '일요일', 1: '월요일', 2: '화요일', 3: '수요일',
           4: '목요일', 5: '금요일', 6: '토요일'}
요일순서 = ['일요일', '월요일', '화요일', '수요일', '목요일', '금요일', '토요일']

df['요일'] = df['order_dow'].map(요일이름)
요일별주문수 = df['요일'].value_counts().reindex(요일순서)

색깔리스트 = []
for 요일 in 요일순서:
    if 요일 == '일요일' or 요일 == '토요일':
        색깔리스트.append('tomato')
    else:
        색깔리스트.append('steelblue')

plt.figure(figsize=(10, 5))
plt.bar(요일순서, 요일별주문수.values, color=색깔리스트)
plt.title('요일별 주문량', fontsize=15)
plt.xlabel('요일')
plt.ylabel('주문 횟수')

for i in range(len(요일순서)):
    숫자 = 요일별주문수.values[i]
    plt.text(i, 숫자 + 500, str(숫자), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('graph1_요일별주문량.png', dpi=150, bbox_inches='tight')
plt.show()

# =============================================
# 2. 시간대별 주문량
# =============================================

시간대별주문수 = df['order_hour_of_day'].value_counts().sort_index()
피크시간 = 시간대별주문수.idxmax()

plt.figure(figsize=(12, 5))
plt.plot(시간대별주문수.index, 시간대별주문수.values, marker='o', color='steelblue', linewidth=2)
plt.axvline(x=피크시간, color='red', linestyle='--', label='피크: ' + str(피크시간) + '시')
plt.axvspan(0, 6, alpha=0.1, color='gray', label='새벽(0~6시)')
plt.title('시간대별 주문량', fontsize=15)
plt.xlabel('시간 (0~23시)')
plt.ylabel('주문 횟수')
plt.xticks(range(0, 24))
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('graph2_시간대별주문량.png', dpi=150, bbox_inches='tight')
plt.show()

# =============================================
# 3. 재구매 주기
# =============================================

재구매데이터 = df['days_since_prior_order'].dropna()
평균값 = 재구매데이터.mean()

print('평균 재구매 주기:', round(평균값, 1), '일')
print('중앙값:', 재구매데이터.median(), '일')
print('최빈값:', 재구매데이터.mode()[0], '일')

plt.figure(figsize=(12, 5))
plt.hist(재구매데이터, bins=30, color='steelblue', edgecolor='white')
plt.axvline(x=평균값, color='red', linestyle='--', linewidth=2,
            label='평균: ' + str(round(평균값, 1)) + '일')
plt.title('재구매 주기 분포', fontsize=15)
plt.xlabel('이전 주문으로부터 경과일 수')
plt.ylabel('주문 횟수')
plt.legend(fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('graph3_재구매주기분포.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
import pandas as pd

# =============================================
# 데이터 불러오기
# =============================================

df = pd.read_csv('../data/orders.csv')

print('데이터 불러오기 완료!')
print('데이터 크기:', df.shape)
print(df.head())


# =============================================
# 메모리 확인 & 최적화
# =============================================

print('최적화 전 메모리 사용량:')
print(round(df.memory_usage(deep=True).sum() / 1024**2, 1), 'MB')

# 데이터 타입을 작은 크기로 줄이기 (메모리 절약)
# int64 → int8, float64 → float32 로 줄여도 값은 그대로
df['order_dow']              = df['order_dow'].astype('int8')
df['order_hour_of_day']      = df['order_hour_of_day'].astype('int8')
df['order_number']           = df['order_number'].astype('int16')
df['days_since_prior_order'] = df['days_since_prior_order'].astype('float32')

print('최적화 후 메모리 사용량:')
print(round(df.memory_usage(deep=True).sum() / 1024**2, 1), 'MB')


# =============================================
# Feature 1 : 유저별 총 주문 건수
#    → EDA에서 주문 횟수가 많을수록 재구매 가능성이 높다고 봤음
# =============================================

총주문건수 = df.groupby('user_id')['order_id'].count()
총주문건수 = 총주문건수.reset_index()
총주문건수.columns = ['user_id', 'total_orders']

print('유저별 총 주문 건수:')
print(총주문건수.head())


# =============================================
# Feature 2 : 유저별 평균 재구매 주기
#    → EDA에서 평균 11.1일, 7일/30일 주기 패턴을 확인했음
#    → 첫 주문은 이전 주문이 없어서 NaN → 제외하고 계산
# =============================================

재구매데이터 = df[df['days_since_prior_order'].notnull()]

재구매빈도 = 재구매데이터.groupby('user_id')['days_since_prior_order'].mean()
재구매빈도 = 재구매빈도.reset_index()
재구매빈도.columns = ['user_id', 'avg_days_between_orders']

print('유저별 평균 재구매 주기:')
print(재구매빈도.head())


# =============================================
# Feature 3 : 유저별 선호 요일 (최빈값)
#    → EDA에서 일요일/월요일에 주문이 집중되는 패턴을 확인했음
#    → 유저마다 자주 주문하는 요일이 다를 수 있으므로 최빈값으로 대표값 추출
# =============================================

가장많은요일 = df.groupby('user_id')['order_dow'].agg(lambda x: x.mode()[0])
가장많은요일 = 가장많은요일.reset_index()
가장많은요일.columns = ['user_id', 'favorite_dow']

print('유저별 선호 요일:')
print(가장많은요일.head())


# =============================================
# Feature 4 : 유저별 선호 시간대 (최빈값)
#    → EDA에서 오전 10시에 주문이 가장 많은 패턴을 확인했음
#    → 유저마다 자주 주문하는 시간대가 다를 수 있으므로 최빈값으로 대표값 추출
# =============================================

가장많은시간 = df.groupby('user_id')['order_hour_of_day'].agg(lambda x: x.mode()[0])
가장많은시간 = 가장많은시간.reset_index()
가장많은시간.columns = ['user_id', 'favorite_hour']

print('유저별 선호 시간대:')
print(가장많은시간.head())


# =============================================
# 전부 합치기 & 메모리 최적화 & 저장
# =============================================

# user_id 기준으로 하나씩 합치기
user_features = 총주문건수.merge(재구매빈도,    on='user_id')
user_features = user_features.merge(가장많은요일, on='user_id')
user_features = user_features.merge(가장많은시간, on='user_id')

# 합친 후에도 메모리 최적화
user_features['total_orders']            = user_features['total_orders'].astype('int16')
user_features['avg_days_between_orders'] = user_features['avg_days_between_orders'].astype('float32')
user_features['favorite_dow']            = user_features['favorite_dow'].astype('int8')
user_features['favorite_hour']           = user_features['favorite_hour'].astype('int8')

print('최종 user_features:')
print(user_features.head(10))
print()
print('크기:', user_features.shape)
print('메모리:', round(user_features.memory_usage(deep=True).sum() / 1024**2, 1), 'MB')

# CSV 저장
user_features.to_csv('../data/user_features.csv', index=False)
print('저장 완료! → user_features.csv')

In [ ]:
import pandas as pd

# =============================================
# 데이터 불러오기
# =============================================

df = pd.read_csv('../data/orders.csv')

print('데이터 불러오기 완료!')
print('데이터 크기:', df.shape)
print(df.head())


# =============================================
# 메모리 확인 & 최적화
# =============================================

print('최적화 전 메모리 사용량:')
print(round(df.memory_usage(deep=True).sum() / 1024**2, 1), 'MB')

# 데이터 타입을 작은 크기로 줄이기 (메모리 절약)
# int64 → int8, float64 → float32 로 줄여도 값은 그대로
df['order_dow']              = df['order_dow'].astype('int8')
df['order_hour_of_day']      = df['order_hour_of_day'].astype('int8')
df['order_number']           = df['order_number'].astype('int16')
df['days_since_prior_order'] = df['days_since_prior_order'].astype('float32')

print('최적화 후 메모리 사용량:')
print(round(df.memory_usage(deep=True).sum() / 1024**2, 1), 'MB')


# =============================================
# Feature 1 : 유저별 총 주문 건수
#    → EDA에서 주문 횟수가 많을수록 재구매 가능성이 높다고 봤음
# =============================================

총주문건수 = df.groupby('user_id')['order_id'].count()
총주문건수 = 총주문건수.reset_index()
총주문건수.columns = ['user_id', 'total_orders']

print('유저별 총 주문 건수:')
print(총주문건수.head())


# =============================================
# Feature 2 : 유저별 평균 재구매 주기
#    → EDA에서 평균 11.1일, 7일/30일 주기 패턴을 확인했음
#    → 첫 주문은 이전 주문이 없어서 NaN → 제외하고 계산
# =============================================

재구매데이터 = df[df['days_since_prior_order'].notnull()]

재구매빈도 = 재구매데이터.groupby('user_id')['days_since_prior_order'].mean()
재구매빈도 = 재구매빈도.reset_index()
재구매빈도.columns = ['user_id', 'avg_days_between_orders']

print('유저별 평균 재구매 주기:')
print(재구매빈도.head())


# =============================================
# Feature 3 : 유저별 선호 요일 (최빈값)
#    → EDA에서 일요일/월요일에 주문이 집중되는 패턴을 확인했음
#    → 유저마다 자주 주문하는 요일이 다를 수 있으므로 최빈값으로 대표값 추출
# =============================================

가장많은요일 = df.groupby('user_id')['order_dow'].agg(lambda x: x.mode()[0])
가장많은요일 = 가장많은요일.reset_index()
가장많은요일.columns = ['user_id', 'favorite_dow']

print('유저별 선호 요일:')
print(가장많은요일.head())


# =============================================
# Feature 4 : 유저별 선호 시간대 (최빈값)
#    → EDA에서 오전 10시에 주문이 가장 많은 패턴을 확인했음
#    → 유저마다 자주 주문하는 시간대가 다를 수 있으므로 최빈값으로 대표값 추출
# =============================================

가장많은시간 = df.groupby('user_id')['order_hour_of_day'].agg(lambda x: x.mode()[0])
가장많은시간 = 가장많은시간.reset_index()
가장많은시간.columns = ['user_id', 'favorite_hour']

print('유저별 선호 시간대:')
print(가장많은시간.head())


# =============================================
# Feature 5 : 재구매 주기의 규칙성 (표준편차) [추가 제안]
#    → 주기가 일정한 유저일수록 다음 주문 시점 예측이 쉬움
#    → 표준편차가 작을수록 규칙적으로 재구매하는 유저
# =============================================

주기규칙성 = 재구매데이터.groupby('user_id')['days_since_prior_order'].std()
주기규칙성 = 주기규칙성.reset_index()
주기규칙성.columns = ['user_id', 'std_days_between_orders']

print('유저별 재구매 주기 규칙성:')
print(주기규칙성.head())


# =============================================
# Feature 6 : 최근 주문 번호 = 충성도 지표 [추가 제안]
#    → order_number가 높을수록 오래된 충성 고객
#    → 충성 고객일수록 재구매 가능성이 높다고 판단
# =============================================

최근주문번호 = df.groupby('user_id')['order_number'].max()
최근주문번호 = 최근주문번호.reset_index()
최근주문번호.columns = ['user_id', 'max_order_number']

print('유저별 최대 주문 번호 (충성도):')
print(최근주문번호.head())


# =============================================
# 전부 합치기 & 메모리 최적화 & 저장
# =============================================

# user_id 기준으로 하나씩 합치기
user_features = 총주문건수.merge(재구매빈도,    on='user_id')
user_features = user_features.merge(가장많은요일,  on='user_id')
user_features = user_features.merge(가장많은시간,  on='user_id')
user_features = user_features.merge(주기규칙성,    on='user_id')
user_features = user_features.merge(최근주문번호,  on='user_id')

# 합친 후에도 메모리 최적화
user_features['total_orders']            = user_features['total_orders'].astype('int16')
user_features['avg_days_between_orders'] = user_features['avg_days_between_orders'].astype('float32')
user_features['favorite_dow']            = user_features['favorite_dow'].astype('int8')
user_features['favorite_hour']           = user_features['favorite_hour'].astype('int8')
user_features['std_days_between_orders'] = user_features['std_days_between_orders'].astype('float32')
user_features['max_order_number']        = user_features['max_order_number'].astype('int16')

print('최종 user_features:')
print(user_features.head(10))
print()
print('크기:', user_features.shape)
print('메모리:', round(user_features.memory_usage(deep=True).sum() / 1024**2, 1), 'MB')

# CSV 저장
user_features.to_csv('../data/user_features.csv', index=False)
print('저장 완료! → user_features.csv')


In [ ]:
import pandas as pd

# =============================================
# 데이터 불러오기
# =============================================

# orders.csv 파일 불러오기
df = pd.read_csv('../data/orders.csv')

# 잘 불러왔는지 확인
print('데이터 불러오기 완료!')
print('데이터 크기:', df.shape)
print(df.head())

# 결측값 확인
print('결측값 확인:')
print(df.isnull().sum())


# =============================================
# 메모리 최적화
# (데이터가 너무 크기 때문에 꼭 필요!)
# =============================================

# 최적화 전에 메모리 얼마나 쓰는지 먼저 확인
before = df.memory_usage(deep=True).sum() / 1024**2
print('최적화 전 메모리:', round(before, 1), 'MB')

# 숫자 범위가 작은 컬럼은 더 작은 타입으로 바꿔주기
# 예) 요일은 0~6밖에 없으니까 int64 쓸 필요 없이 int8로 충분
df['order_dow']              = df['order_dow'].astype('int8')
df['order_hour_of_day']      = df['order_hour_of_day'].astype('int8')
df['order_number']           = df['order_number'].astype('int16')
df['days_since_prior_order'] = df['days_since_prior_order'].astype('float32')

# 최적화 후 메모리 얼마나 줄었는지 확인
after = df.memory_usage(deep=True).sum() / 1024**2
print('최적화 후 메모리:', round(after, 1), 'MB')
print('줄어든 양:', round(before - after, 1), 'MB')


# =============================================
# Feature 1 : 유저별 총 주문 건수
# → 주문을 많이 한 유저일수록 재구매할 가능성이 높다고 생각함
# =============================================

# user_id 기준으로 묶어서 주문 횟수 세기
총주문건수 = df.groupby('user_id')['order_id'].count()

# 인덱스 초기화해서 보기 좋게 만들기
총주문건수 = 총주문건수.reset_index()

# 컬럼 이름 바꾸기
총주문건수.columns = ['user_id', 'total_orders']

# 잘 만들어졌는지 확인
print('유저별 총 주문 건수:')
print(총주문건수.head())
print('전체 유저 수:', len(총주문건수))


# =============================================
# Feature 2 : 유저별 평균 재구매 주기
# → EDA 에서 평균 11.1일 주기 확인했음
# → 첫 주문은 이전 주문이 없으니까 NaN → 빼고 계산
# =============================================

# NaN 인 행 제거 (= 첫 주문 제거)
재구매데이터 = df[df['days_since_prior_order'].notnull()]
print('첫 주문 제거 후 행 수:', len(재구매데이터))

# user_id 기준으로 묶어서 평균 계산
재구매빈도 = 재구매데이터.groupby('user_id')['days_since_prior_order'].mean()
재구매빈도 = 재구매빈도.reset_index()
재구매빈도.columns = ['user_id', 'avg_days_between_orders']

print('유저별 평균 재구매 주기:')
print(재구매빈도.head())


# =============================================
# Feature 3 : 유저별 선호 요일
# → EDA 에서 일요일/월요일에 주문 집중 확인했음
# → 유저마다 자주 주문하는 요일이 다르니까 최빈값으로 뽑기
# =============================================

# mode() = 가장 많이 나온 값 (최빈값)
# [0] 은 최빈값이 여러 개일 때 첫 번째 것만 쓰려고
가장많은요일 = df.groupby('user_id')['order_dow'].agg(lambda x: x.mode()[0])
가장많은요일 = 가장많은요일.reset_index()
가장많은요일.columns = ['user_id', 'favorite_dow']

print('유저별 선호 요일:')
print(가장많은요일.head())


# =============================================
# Feature 4 : 유저별 선호 시간대
# → EDA 에서 오전 10시 피크 확인했음
# → 유저마다 자주 주문하는 시간대가 다르니까 최빈값으로 뽑기
# =============================================

가장많은시간 = df.groupby('user_id')['order_hour_of_day'].agg(lambda x: x.mode()[0])
가장많은시간 = 가장많은시간.reset_index()
가장많은시간.columns = ['user_id', 'favorite_hour']

print('유저별 선호 시간대:')
print(가장많은시간.head())


# =============================================
# Feature 5 : 재구매 주기 규칙성 [추가 제안]
# → 주기가 일정한 유저일수록 다음 주문 시점 예측이 쉬움
# → 표준편차가 작을수록 규칙적으로 재구매하는 유저
# =============================================

주기규칙성 = 재구매데이터.groupby('user_id')['days_since_prior_order'].std()
주기규칙성 = 주기규칙성.reset_index()
주기규칙성.columns = ['user_id', 'std_days_between_orders']

print('유저별 재구매 주기 규칙성:')
print(주기규칙성.head())


# =============================================
# Feature 6 : 충성도 지표 [추가 제안]
# → order_number 가 높을수록 오래된 충성 고객
# → 충성 고객일수록 재구매 가능성이 높다고 판단
# =============================================

최근주문번호 = df.groupby('user_id')['order_number'].max()
최근주문번호 = 최근주문번호.reset_index()
최근주문번호.columns = ['user_id', 'max_order_number']

print('유저별 충성도 지표:')
print(최근주문번호.head())


# =============================================
# 전부 합치기
# → user_id 기준으로 하나씩 붙이기
# =============================================

user_features = 총주문건수.merge(재구매빈도,   on='user_id')
user_features = user_features.merge(가장많은요일, on='user_id')
user_features = user_features.merge(가장많은시간, on='user_id')
user_features = user_features.merge(주기규칙성,   on='user_id')
user_features = user_features.merge(최근주문번호, on='user_id')

# 합쳐진 결과 확인
print('합친 후 user_features:')
print(user_features.head(10))
print('크기:', user_features.shape)


# =============================================
# 합친 후 메모리 최적화
# =============================================

user_features['total_orders']            = user_features['total_orders'].astype('int16')
user_features['avg_days_between_orders'] = user_features['avg_days_between_orders'].astype('float32')
user_features['favorite_dow']            = user_features['favorite_dow'].astype('int8')
user_features['favorite_hour']           = user_features['favorite_hour'].astype('int8')
user_features['std_days_between_orders'] = user_features['std_days_between_orders'].astype('float32')
user_features['max_order_number']        = user_features['max_order_number'].astype('int16')

print('최종 메모리:', round(user_features.memory_usage(deep=True).sum() / 1024**2, 1), 'MB')


# =============================================
# CSV 저장
# =============================================

user_features.to_csv('../data/user_features.csv', index=False)
print('저장 완료! → user_features.csv')


In [ ]:
import pandas as pd

# =============================================
# 데이터 불러오기
# =============================================

# orders.csv 파일 불러오기
df = pd.read_csv('../data/orders.csv')

# 잘 불러왔는지 확인
print('데이터 불러오기 완료!')
print('데이터 크기:', df.shape)
print(df.head())

# 결측값 확인
print('결측값 확인:')
print(df.isnull().sum())


# =============================================
# 메모리 최적화
# (데이터가 너무 크기 때문에 꼭 필요!)
# =============================================

# 최적화 전에 메모리 얼마나 쓰는지 먼저 확인
before = df.memory_usage(deep=True).sum() / 1024**2
print('최적화 전 메모리:', round(before, 1), 'MB')

# 숫자 범위가 작은 컬럼은 더 작은 타입으로 바꿔주기
# 예) 요일은 0~6밖에 없으니까 int64 쓸 필요 없이 int8로 충분
df['order_dow']              = df['order_dow'].astype('int8')
df['order_hour_of_day']      = df['order_hour_of_day'].astype('int8')
df['order_number']           = df['order_number'].astype('int16')
df['days_since_prior_order'] = df['days_since_prior_order'].astype('float32')

# 최적화 후 메모리 얼마나 줄었는지 확인
after = df.memory_usage(deep=True).sum() / 1024**2
print('최적화 후 메모리:', round(after, 1), 'MB')
print('줄어든 양:', round(before - after, 1), 'MB')


# =============================================
# Feature 1 : 유저별 총 주문 건수
# → 주문을 많이 한 유저일수록 재구매할 가능성이 높다고 생각함
# =============================================

# user_id 기준으로 묶어서 주문 횟수 세기
총주문건수 = df.groupby('user_id')['order_id'].count()

# 인덱스 초기화해서 보기 좋게 만들기
총주문건수 = 총주문건수.reset_index()

# 컬럼 이름 바꾸기
총주문건수.columns = ['user_id', 'total_orders']

# 잘 만들어졌는지 확인
print('유저별 총 주문 건수:')
print(총주문건수.head())
print('전체 유저 수:', len(총주문건수))


# =============================================
# Feature 2 : 유저별 평균 재구매 주기
# → EDA 에서 평균 11.1일 주기 확인했음
# → 첫 주문은 이전 주문이 없으니까 NaN → 빼고 계산
# =============================================

# NaN 인 행 제거 (= 첫 주문 제거)
재구매데이터 = df[df['days_since_prior_order'].notnull()]
print('첫 주문 제거 후 행 수:', len(재구매데이터))

# user_id 기준으로 묶어서 평균 계산
재구매빈도 = 재구매데이터.groupby('user_id')['days_since_prior_order'].mean()
재구매빈도 = 재구매빈도.reset_index()
재구매빈도.columns = ['user_id', 'avg_days_between_orders']

print('유저별 평균 재구매 주기:')
print(재구매빈도.head())


# =============================================
# Feature 3 : 유저별 선호 요일
# → EDA 에서 일요일/월요일에 주문 집중 확인했음
# → 유저마다 자주 주문하는 요일이 다르니까 최빈값으로 뽑기
# =============================================

# mode() = 가장 많이 나온 값 (최빈값)
# [0] 은 최빈값이 여러 개일 때 첫 번째 것만 쓰려고
가장많은요일 = df.groupby('user_id')['order_dow'].agg(lambda x: x.mode()[0])
가장많은요일 = 가장많은요일.reset_index()
가장많은요일.columns = ['user_id', 'favorite_dow']

print('유저별 선호 요일:')
print(가장많은요일.head())


# =============================================
# Feature 4 : 유저별 선호 시간대
# → EDA 에서 오전 10시 피크 확인했음
# → 유저마다 자주 주문하는 시간대가 다르니까 최빈값으로 뽑기
# =============================================

가장많은시간 = df.groupby('user_id')['order_hour_of_day'].agg(lambda x: x.mode()[0])
가장많은시간 = 가장많은시간.reset_index()
가장많은시간.columns = ['user_id', 'favorite_hour']

print('유저별 선호 시간대:')
print(가장많은시간.head())


# =============================================
# Feature 5 : 재구매 주기 규칙성 [추가 제안]
# → 주기가 일정한 유저일수록 다음 주문 시점 예측이 쉬움
# → 표준편차가 작을수록 규칙적으로 재구매하는 유저
# =============================================

주기규칙성 = 재구매데이터.groupby('user_id')['days_since_prior_order'].std()
주기규칙성 = 주기규칙성.reset_index()
주기규칙성.columns = ['user_id', 'std_days_between_orders']

print('유저별 재구매 주기 규칙성:')
print(주기규칙성.head())


# =============================================
# Feature 6 : 충성도 지표 [추가 제안]
# → order_number 가 높을수록 오래된 충성 고객
# → 충성 고객일수록 재구매 가능성이 높다고 판단
# =============================================

최근주문번호 = df.groupby('user_id')['order_number'].max()
최근주문번호 = 최근주문번호.reset_index()
최근주문번호.columns = ['user_id', 'max_order_number']

print('유저별 충성도 지표:')
print(최근주문번호.head())


# =============================================
# 전부 합치기
# → user_id 기준으로 하나씩 붙이기
# =============================================

user_features = 총주문건수.merge(재구매빈도,   on='user_id')
user_features = user_features.merge(가장많은요일, on='user_id')
user_features = user_features.merge(가장많은시간, on='user_id')
user_features = user_features.merge(주기규칙성,   on='user_id')
user_features = user_features.merge(최근주문번호, on='user_id')

# 합쳐진 결과 확인
print('합친 후 user_features:')
print(user_features.head(10))
print('크기:', user_features.shape)


# =============================================
# 합친 후 메모리 최적화
# =============================================

user_features['total_orders']            = user_features['total_orders'].astype('int16')
user_features['avg_days_between_orders'] = user_features['avg_days_between_orders'].astype('float32')
user_features['favorite_dow']            = user_features['favorite_dow'].astype('int8')
user_features['favorite_hour']           = user_features['favorite_hour'].astype('int8')
user_features['std_days_between_orders'] = user_features['std_days_between_orders'].astype('float32')
user_features['max_order_number']        = user_features['max_order_number'].astype('int16')

print('최종 메모리:', round(user_features.memory_usage(deep=True).sum() / 1024**2, 1), 'MB')


# =============================================
# CSV 저장
# =============================================

user_features.to_csv('../data/user_features.csv', index=False)
print('저장 완료! → user_features.csv')
